# Order Nakeds

In [1]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "NSE"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.precision', 2)

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

1

## Imports

In [2]:
from utils import handle_raws, get_pickle, load_config, arrange_orders, pickle_me, get_file_age, yes_or_no
from datetime import datetime
from ib_async import IB
from ibfuncs import get_open_orders, quick_pf, place_orders, make_ib_orders

## Set constants

In [3]:
config = load_config(MARKET)
port = config.get('PORT')
MARGINPERORDER = config.get('MARGINPERORDER')

## Handle Raws

In [4]:
# Consolidate raw files to df_nakeds.pkl
pattern = str(f"*{MARKET.lower()}nakeds*.pkl")

handle_raws(pattern=pattern)

2024-08-14 06:44:20.764 | INFO     | utils:delete_files:194 - Deleted: C:\Users\kashi\python\nse\data\raw\nsenakeds1.pkl
2024-08-14 06:44:20.765 | INFO     | utils:remove_raw_nakeds:232 - Deleted files [WindowsPath('C:/Users/kashi/python/nse/data/raw/nsenakeds1.pkl')]


In [5]:
file_path = ROOT / 'data' / 'df_nakeds.pkl'

def how_many_days_old(file_path) -> float:
    """Gets the file's age in days"""
    file_age = get_file_age(file_path=file_path)
    
    seconds_in_a_day = 86400
    file_age_in_days = file_age.td.total_seconds() / seconds_in_a_day if file_age else 0
    
    return file_age_in_days

In [12]:
## Check the age of df_pickles, before ordering
txt = f"df_nakeds.pkl is {how_many_days_old(file_path): 0.2f}. Want to load?"
ans = yes_or_no(txt)

if ans:
    df_opts = get_pickle(file_path)
    print('\n\n')
    print(df_opts.drop(columns=['nse_symbol', 'instrument', 'contract', 'expiry']).head())
else:
    print('Bye!!!')




   ib_symbol   undPrice  safe_strike right    strike    dte    hv    iv   lot  \
0   TATACHEM    1025.55         1135     C    1140.0  15.47  0.41  0.35   550   
1        MRF  137528.40       150918     C  154000.0  15.48  0.28  0.32     5   
2     AUBANK     610.10          670     C     670.0  15.47  0.37  0.32  1000   
3  TATAPOWER     408.00          458     C     460.0  15.48  0.43  0.40  1350   
4     ASTRAL    1909.15         2140     C    2200.0  15.47  0.35  0.39   367   

    price     sdev  intrinsic  bsPrice  comm   margin  xPrice    rom  
0    2.90    73.26        0.0     2.59  20.0  6888.67    2.90   5.46  
1  198.70  8926.70        0.0   173.33  20.0  1867.38  198.70  12.55  
2    1.85    40.39        0.0     1.65  20.0  8679.95    1.85   5.03  
3    1.40    33.69        0.0     1.26  20.0  5543.21    1.40   8.04  
4    3.30   154.40        0.0     2.90  20.0  2388.56    3.30  11.96  


## check open orders

In [13]:
# check open orders
with IB().connect(port=port, clientId=10) as ib:
    dfo = get_open_orders(ib)
    dfp = quick_pf(ib)
    

In [14]:
if not dfo.empty:
    remove_opens = set(dfo.symbol.to_list())
else:
    remove_opens = set()

In [ ]:
# make a list of symbols to be removed from df_opts

if not dfp.empty:
    remove_positions = set(dfp.symbol.to_list())
else:
    remove_positions = set()

remove_ib_syms = remove_opens | remove_positions

# get the target options to plant
dft = df_opts[~df_opts.ib_symbol.isin(remove_ib_syms)].reset_index(drop=True)

In [16]:
len(dft)

1294

## Arrange and make orders

In [17]:
df_nakeds = arrange_orders(dft, maxmargin=MARGINPERORDER)
cos = make_ib_orders(df_nakeds)

## PLACE THE ORDER

# Archive the orders into `xn_history`

In [19]:
filename = f"{datetime.now().strftime('%Y%m%d_%I_%M_%p')}_naked_orders.pkl"
pickle_me(ordered, str(ROOT / "data" / "xn_history" / str(filename)))